In [1]:
import dimod
from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.samplers import SimulatedAnnealingSampler
import networkx as nx
import numpy as np
import csv

In [2]:

# ============================================================
# 1. GLOBAL PARAMETERS
# ============================================================

START_YEAR = 2026
END_YEAR = 2040

A_START = 0.0
A_GROWTH = 0.07

TIE_SCALE = 0.15
REGIONAL_TIE = 0.05

NOISE_AMPLITUDE = 0.02
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)


# ============================================================
# 2. INDUSTRY RESISTANCE
# ============================================================

base_resistance = {
    "Technology": 0.10,
    "Media": 0.20,
    "Telecom": 0.25,
    "Retail": 0.30,
    "Hospitality": 0.35,
    "Logistics": 0.40,
    "Banking": 0.45,
    "Accounting": 0.50,
    "Manufacturing": 0.50,
    "Insurance": 0.55,
    "Energy": 0.55,
    "Pharma": 0.60,
    "Education": 0.60,
    "Nonprofit": 0.65,
    "Real Estate": 0.65,
    "Construction": 0.70,
    "Agriculture": 0.70,
    "Healthcare": 0.75,
    "Government": 0.85,
    "Legal": 0.90,
}


# ============================================================
# 3. REGIONS AND REGIONAL RESISTANCE SHIFTS
# ============================================================

regions = {
    "Pacific Northwest": -0.049,
    "California": -0.060,
    "Mountain West": 0.076,
    "Southwest": -0.012,
    "Texas": 0.072,
    "Great Plains": 0.077,
    "Midwest": 0.027,
    "Great Lakes": 0.011,
    "Lower South": 0.083,
    "Upper South": 0.027,
    "Mid-Atlantic": -0.048,
    "Northeast": -0.046,
    "Alaska": 0
}



# ============================================================
# 4. CREATE INDUSTRY-REGION NODES
# ============================================================

nodes = [
    (industry, region)
    for region in regions
    for industry in base_resistance
]


# Each node receives its own resistance value

resistance = {}

for industry, region in nodes:

    base = base_resistance[industry]
    regional_shift = regions[region]

    noise = rng.uniform(
        -NOISE_AMPLITUDE,
        NOISE_AMPLITUDE
    )

    resistance[(industry, region)] = np.clip(
        base + (2*regional_shift) + noise,
        0.0,
        1.0
    )


# ============================================================
# 5. INDUSTRY PARTNER TIES
# ============================================================

raw_ties = [
    ("Healthcare", "Insurance", 1.0),
    ("Manufacturing", "Logistics", 0.9),
    ("Banking", "Real Estate", 0.8),
    ("Retail", "Logistics", 0.8),
    ("Construction", "Real Estate", 0.8),
    ("Healthcare", "Pharma", 0.7),
    ("Legal", "Real Estate", 0.7),
    ("Healthcare", "Government", 0.6),
    ("Legal", "Government", 0.6),
    ("Accounting", "Banking", 0.6),
    ("Government", "Education", 0.6),
    ("Agriculture", "Logistics", 0.6),
    ("Telecom", "Technology", 0.6),
    ("Insurance", "Banking", 0.5),
    ("Legal", "Banking", 0.5),
    ("Energy", "Manufacturing", 0.5),
    ("Technology", "Media", 0.5),
    ("Insurance", "Legal", 0.4),
    ("Accounting", "Nonprofit", 0.4),
    ("Construction", "Government", 0.4),
    ("Hospitality", "Retail", 0.4),
    ("Pharma", "Logistics", 0.4),
    ("Nonprofit", "Education", 0.4),
    ("Media", "Retail", 0.3),
]


# Scale partner ties

partner_ties = [
    (i, j, strength * TIE_SCALE)
    for i, j, strength in raw_ties
]


# ============================================================
# 6. GEOGRAPHICALLY NEIGHBORING REGIONS
# ============================================================

regional_connections = [
    ("Pacific Northwest", "California"),
    ("Pacific Northwest", "Mountain West"),
    ("California", "Mountain West"),
    ("California", "Southwest"),
    ("Mountain West", "Southwest"),
    ("Mountain West", "Great Plains"),
    ("Southwest", "Texas"),
    ("Southwest", "Great Plains"),
    ("Texas", "Great Plains"),
    ("Texas", "Lower South"),
    ("Great Plains", "Midwest"),
    ("Midwest", "Great Lakes"),
    ("Midwest", "Lower South"),
    ("Great Lakes", "Upper South"),
    ("Great Lakes", "Mid-Atlantic"),
    ("Lower South", "Upper South"),
    ("Upper South", "Mid-Atlantic"),
    ("Mid-Atlantic", "Northeast"),
]


# ============================================================
# 7. BUILD THE FULL NETWORK
# ============================================================

G = nx.DiGraph()

SOURCE = "__SOURCE__"
SINK = "__SINK__"

G.add_nodes_from(nodes)

# Within-region industry relationships

for region in regions:

    for industry1, industry2, J in partner_ties:

        node1 = (industry1, region)
        node2 = (industry2, region)

        G.add_edge(node1, node2, tie=J)
        G.add_edge(node2, node1, tie=J)


# Between-region relationships
# Same industry in neighboring regions

for region1, region2 in regional_connections:

    for industry in base_resistance:

        node1 = (industry, region1)
        node2 = (industry, region2)

        G.add_edge(
            node1, node2,
            tie=REGIONAL_TIE
        )

        G.add_edge(
            node2, node1,
            tie=REGIONAL_TIE
        )


In [3]:
# ============================================================
# 8. MINIMUM CUT SOLVER
# ============================================================

def solve_year(A):

    H = nx.DiGraph()

    H.add_nodes_from(G.nodes)

    # Add unary resistance / AI pressure costs

    for node in nodes:

        h = A - resistance[node]

        cost_paper = max(h, 0)
        cost_digital = max(-h, 0)

        H.add_edge(
            SOURCE,
            node,
          #  capacity=cost_paper
          capacity = cost_digital
        )

        H.add_edge(
            node,
            SINK,
         #   capacity=cost_digital
         capacity = cost_paper
        )

    # Add all network coupling strengths

    for node1, node2 in G.edges:

        H.add_edge(
            node1,
            node2,
            capacity=G[node1][node2]["tie"]
        )

    # Find global minimum energy

    cut_value, partition = nx.minimum_cut(
        H,
        SOURCE,
        SINK
    )

    source_side, sink_side = partition

    states = {}

    for node in nodes:

        if node in source_side:
            states[node] = 1

        else:
            states[node] = -1

    return states, cut_value


# ============================================================
# 9. SIMULATE 2026-2040
# ============================================================

results = {}
cut_values = {}

for year in range(START_YEAR, END_YEAR + 1):

    A = A_START + A_GROWTH * (year - START_YEAR)

    states, cut = solve_year(A)

    results[year] = states
    cut_values[year] = cut

In [4]:
# Dictionaries to store results across all years
results = {}
cut_values = {}

for year in range(START_YEAR, END_YEAR + 1):
    # 1. Calculate the incremental AI pressure for the current year
    A = A_START + A_GROWTH * (year - START_YEAR)
    
    # 2. Solve the network state for this year's pressure level
    states, cut_value = solve_year(A)
    
    # 3. Save the outputs
    results[year] = states
    cut_values[year] = cut_value
    
    # Optional: Print a quick update to track progress
    print(f"Completed year {year} (A = {A:.2f}) - Energy: {cut_value:.2f}")

Completed year 2026 (A = 0.00) - Energy: 0.00
Completed year 2027 (A = 0.07) - Energy: 0.25
Completed year 2028 (A = 0.14) - Energy: 0.79
Completed year 2029 (A = 0.21) - Energy: 1.38
Completed year 2030 (A = 0.28) - Energy: 1.92
Completed year 2031 (A = 0.35) - Energy: 2.06
Completed year 2032 (A = 0.42) - Energy: 2.98
Completed year 2033 (A = 0.49) - Energy: 4.16
Completed year 2034 (A = 0.56) - Energy: 6.21
Completed year 2035 (A = 0.63) - Energy: 7.86
Completed year 2036 (A = 0.70) - Energy: 8.20
Completed year 2037 (A = 0.77) - Energy: 4.77
Completed year 2038 (A = 0.84) - Energy: 2.22
Completed year 2039 (A = 0.91) - Energy: 0.81
Completed year 2040 (A = 0.98) - Energy: 0.13


In [5]:
# Extract unique regions alphabetically
unique_regions = sorted(list(set(region for industry, region in nodes)))

print("==================================================")
print("     REGIONAL AI ADOPTION PROGRESS (2026-2040)    ")
print("==================================================")

for year in range(START_YEAR, END_YEAR + 1):
    print(f"\n📅 Year: {year} (AI Pressure A = {A_START + A_GROWTH * (year - START_YEAR):.2f})")
    year_states = results[year]
    
    for region in unique_regions:
        # Get all industry nodes belonging to this region
        region_nodes = [node for node in nodes if node[1] == region]
        
        # Count how many industries in this region have adopted digital (-1)
        digital_count = sum(1 for node in region_nodes if year_states[node] == -1)
        total_industries = len(region_nodes)
        
        # Calculate percentage
        adoption_pct = (digital_count / total_industries) * 100
        
        # Visual progress bar (optional)
        bar = "█" * int(adoption_pct // 10) + "-" * (10 - int(adoption_pct // 10))
        
        print(f"  {region:<18} | [{bar}] {adoption_pct:5.1f}% digital ({digital_count}/{total_industries})")

     REGIONAL AI ADOPTION PROGRESS (2026-2040)    

📅 Year: 2026 (AI Pressure A = 0.00)
  Alaska             | [----------]   0.0% digital (0/20)
  California         | [----------]   0.0% digital (0/20)
  Great Lakes        | [----------]   0.0% digital (0/20)
  Great Plains       | [----------]   0.0% digital (0/20)
  Lower South        | [----------]   0.0% digital (0/20)
  Mid-Atlantic       | [----------]   0.0% digital (0/20)
  Midwest            | [----------]   0.0% digital (0/20)
  Mountain West      | [----------]   0.0% digital (0/20)
  Northeast          | [----------]   0.0% digital (0/20)
  Pacific Northwest  | [----------]   0.0% digital (0/20)
  Southwest          | [----------]   0.0% digital (0/20)
  Texas              | [----------]   0.0% digital (0/20)
  Upper South        | [----------]   0.0% digital (0/20)

📅 Year: 2027 (AI Pressure A = 0.07)
  Alaska             | [----------]   0.0% digital (0/20)
  California         | [----------]   0.0% digital (0/20)
  Gre

In [6]:
#import csv

# Define your custom column order for regions
target_regions = [
    "Pacific Northwest", "California", "Mountain West", "Southwest", 
    "Texas", "Great Plains", "Midwest", "Great Lakes", 
    "Lower South", "Upper South", "Mid-Atlantic", "Northeast", "Alaska"
]

# Get unique industries from your existing base_resistance dictionary
unique_industries = list(base_resistance.keys())
filename = "data_without_regional_ties.csv"

# Open the file and write the cross-sectional matrix
with open(filename, mode="w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    
    # Write header row
    writer.writerow(["Industry"] + target_regions)
    
    # Write each industry row
    for industry in unique_industries:
        row_data = [industry]
        for region in target_regions:
            node = (industry, region)
            adopted_year = None
            
            # Scan through years to find the first transition to digital (-1)
            for year in range(START_YEAR, END_YEAR + 1):
                if results.get(year, {}).get(node) == -1:
                    adopted_year = float(year)
                    break
            
            # Format as float string if adopted, otherwise leave blank
            formatted_val = f"{adopted_year:.1f}" if adopted_year is not None else ""
            row_data.append(formatted_val)
        
        writer.writerow(row_data)

print(f"Successfully generated and saved cross-sectional matrix to {filename}!")

Successfully generated and saved cross-sectional matrix to data_without_regional_ties.csv!
